# ARC-AGI-3 — graph explorer + CHIMERA world model (Agnuxo1)

Policy `chimera`, options `{"use_kbca": true, "alpha": 4.0}`. Non-LLM agent: state-graph exploration; CHIMERA (holographic phase memory + local cellular-automaton rules, numpy port of github.com/Agnuxo1/ARC2_CHIMERA) re-ranks untried actions. Apache-2.0.


In [ ]:
import time
T_KERNEL = time.time()   # global clock starts with the kernel
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi==0.9.8 python-dotenv


In [ ]:
import os
os.makedirs('/kaggle/working/arc3chimera', exist_ok=True)


In [ ]:
%%writefile /kaggle/working/arc3chimera/runner.py
"""Local evaluation with the official arc_agi scorecard (offline environment files).

Each (game, seed) runs in its own process with its own Arcade, so the
scorecard of that process holds exactly one play of one game. The agent only
sees the public observation (frames, state, level counter, available actions).
"""
from __future__ import annotations

import argparse
import json
import logging
import os
import time
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np

logging.disable(logging.INFO)


def make_policy(name: str, seed: int, **kw):
    if name == "explorer":
        from .explorer import GraphExplorer
        return GraphExplorer(seed=seed)
    if name == "chimera":
        from .chimera import ChimeraAgent
        return ChimeraAgent(seed=seed, **kw)
    if name == "random":
        from .baselines import RandomAgent
        return RandomAgent(seed=seed)
    raise ValueError(name)


def play_one(env_dir: str, game_id: str, policy_name: str, seed: int, max_actions: int,
             max_seconds: float, kw: dict | None = None) -> dict:
    import arc_agi
    from arcengine import GameAction, GameState

    from .explorer import Obs
    from .perception import last_grid

    logging.disable(logging.CRITICAL)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    env = arcade.make(game_id, seed=seed)
    policy = make_policy(policy_name, seed, **(kw or {}))
    r = env.observation_space
    t0 = time.time()
    n = 0
    level_actions = []
    last_levels = 0
    plays = 1
    while r is not None and r.state != GameState.WIN and n < max_actions and time.time() - t0 < max_seconds:
        obs = Obs(grid=last_grid(r.frame), state=r.state.name, levels=r.levels_completed,
                  available=list(r.available_actions))
        aid, data = policy.act(obs)
        r = env.step(GameAction.from_id(aid), data)
        n += 1
        if r is not None and r.levels_completed != last_levels:
            level_actions.append(n)
            last_levels = r.levels_completed
    sc = arcade.get_scorecard()
    env_scores = [e for e in sc.environments]
    rec = env_scores[0] if env_scores else None
    run_list = rec.runs if rec is not None and hasattr(rec, "runs") else []
    out = {
        "game": game_id, "policy": policy_name, "seed": seed,
        "score": float(rec.score) if rec is not None else 0.0,
        "levels": int(rec.levels_completed) if rec is not None else 0,
        "actions": n, "plays": len(run_list) if run_list else plays,
        "level_actions": level_actions,
        "level_scores": list(run_list[0].level_scores) if run_list and run_list[0].level_scores else [],
        "baseline": list(run_list[0].level_baseline_actions) if run_list and run_list[0].level_baseline_actions else [],
        "seconds": round(time.time() - t0, 2),
        "stats": getattr(policy, "stats", {}),
    }
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--env-dir", required=True)
    ap.add_argument("--policy", default="explorer")
    ap.add_argument("--seeds", default="0")
    ap.add_argument("--games", default="")
    ap.add_argument("--max-actions", type=int, default=3000)
    ap.add_argument("--max-seconds", type=float, default=600)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--out", required=True)
    ap.add_argument("--kw", default="{}")
    a = ap.parse_args()
    import arc_agi
    logging.disable(logging.CRITICAL)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=a.env_dir)
    games = [e.game_id for e in arcade.available_environments]
    if a.games:
        sel = a.games.split(",")
        games = [g for g in games if g.split("-")[0] in sel]
    seeds = [int(s) for s in a.seeds.split(",")]
    kw = json.loads(a.kw)
    jobs = [(g, s) for s in seeds for g in games]
    results = []
    t0 = time.time()
    with ProcessPoolExecutor(max_workers=a.workers) as ex:
        futs = {ex.submit(play_one, a.env_dir, g, a.policy, s, a.max_actions, a.max_seconds, kw): (g, s) for g, s in jobs}
        for f in as_completed(futs):
            g, s = futs[f]
            try:
                res = f.result()
            except Exception as e:  # keep the benchmark alive, record the failure
                res = {"game": g, "policy": a.policy, "seed": s, "score": 0.0, "levels": 0, "error": repr(e)}
            results.append(res)
            print(json.dumps({k: res.get(k) for k in ("game", "seed", "score", "levels", "actions", "plays", "seconds", "error")}), flush=True)
    per_seed = {}
    for r in results:
        per_seed.setdefault(r["seed"], []).append(r["score"])
    seed_means = {s: float(np.mean(v)) for s, v in per_seed.items()}
    summary = {
        "policy": a.policy, "kw": kw, "max_actions": a.max_actions, "games": len(games), "seeds": seeds,
        "mean_score": float(np.mean(list(seed_means.values()))),
        "std_score_over_seeds": float(np.std(list(seed_means.values()))),
        "seed_means": seed_means,
        "levels_total_per_seed": {s: int(sum(r["levels"] for r in results if r["seed"] == s)) for s in seeds},
        "errors": sum(1 for r in results if "error" in r),
        "extra_plays": sum(1 for r in results if r.get("plays", 1) > 1),
        "wall_seconds": round(time.time() - t0, 1),
    }
    os.makedirs(os.path.dirname(os.path.abspath(a.out)), exist_ok=True)
    with open(a.out, "w", encoding="utf-8") as f:
        json.dump({"summary": summary, "results": sorted(results, key=lambda r: (r["seed"], r["game"]))}, f, indent=1)
    print(json.dumps(summary))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/arc3chimera/__init__.py
"""ARC-AGI-3 agents: graph-exploration control and CHIMERA world-model agent."""
__version__ = "0.1.0"


In [ ]:
%%writefile /kaggle/working/arc3chimera/perception.py
"""Observation-only perception for ARC-AGI-3 frames.

Everything here uses only what the public API returns (the rendered frames,
game state, level counter and available actions). No engine internals.
"""
from __future__ import annotations

import hashlib
from dataclasses import dataclass

import numpy as np

GRID = 64


def last_grid(frame_list) -> np.ndarray:
    """Return the settled (last) grid of an observation as int8 64x64."""
    if frame_list is None or len(frame_list) == 0:
        return np.zeros((GRID, GRID), dtype=np.int8)
    g = np.asarray(frame_list[-1], dtype=np.int8)
    if g.shape != (GRID, GRID):
        out = np.zeros((GRID, GRID), dtype=np.int8)
        h, w = min(GRID, g.shape[0]), min(GRID, g.shape[1])
        out[:h, :w] = g[:h, :w]
        g = out
    return g


def grid_hash(g: np.ndarray, mask: np.ndarray | None = None) -> bytes:
    if mask is not None and mask.any():
        g = g.copy()
        g[mask] = -1
    return hashlib.blake2b(g.tobytes(), digest_size=12).digest()


class VolatilityTracker:
    """Detects housekeeping pixels (step counters, energy bars, timers).

    A pixel is volatile when it changes in a large share of the transitions
    in which anything changed. Those pixels are masked out of the state key so
    that a moving counter does not make every state look new.
    """

    def __init__(self, min_transitions: int = 6, ratio: float = 0.6, border: int = 4,
                 border_ratio: float = 0.2, lines: bool = True, min_line: int = 3):
        self.min_transitions = min_transitions
        self.ratio = ratio
        self.border = border
        self.border_ratio = border_ratio
        # progress bars grow one pixel per action: each pixel changes once, but the
        # border line holding the bar changes on almost every transition
        self.lines = lines
        self.min_line = min_line
        self.reset()
        b = border
        self._border = np.zeros((GRID, GRID), dtype=bool)
        self._border[:b, :] = True
        self._border[-b:, :] = True
        self._border[:, :b] = True
        self._border[:, -b:] = True

    def reset(self):
        self.counts = np.zeros((GRID, GRID), dtype=np.int32)
        self.n = 0
        self.mask = np.zeros((GRID, GRID), dtype=bool)
        self.version = 0
        self.row_counts = np.zeros(GRID, dtype=np.int32)
        self.col_counts = np.zeros(GRID, dtype=np.int32)

    def _line_mask(self) -> np.ndarray | None:
        b, m = self.border, np.zeros((GRID, GRID), dtype=bool)
        edge = np.r_[0:b, GRID - b:GRID]
        for i in edge:
            if self.row_counts[i] >= self.ratio * self.n:
                m[i, :] = True
            if self.col_counts[i] >= self.ratio * self.n:
                m[:, i] = True
        return m

    def update(self, before: np.ndarray, after: np.ndarray) -> bool:
        diff = before != after
        if not diff.any():
            return False
        self.counts += diff
        self.n += 1
        if self.lines:
            small = diff.sum() <= 2 * GRID       # a bar step, not a full-screen redraw
            if small:
                self.row_counts += diff.any(axis=1)
                self.col_counts += diff.any(axis=0)
            if self.n >= self.min_line:
                lm = self._line_mask() & ~self.mask
                if lm.any() and (self.mask | lm).mean() <= 0.33:
                    self.mask = self.mask | lm
                    self.version += 1
                    return True
        if self.n < self.min_transitions:
            return False
        if self.n % 3 and self.version:
            return False
        share = self.counts / self.n
        cand = (share >= self.ratio) | (self._border & (share >= self.border_ratio))
        # never mask more than a third of the screen: that would be the game itself
        if cand.mean() > 0.33:
            cand = share >= max(self.ratio, 0.9)
        new_mask = self.mask | cand          # monotonic: avoids re-keying back and forth
        if new_mask.mean() > 0.33:
            return False
        if (new_mask != self.mask).any():
            self.mask = new_mask
            self.version += 1
            return True
        return False


@dataclass
class Segment:
    color: int
    size: int
    cy: int
    cx: int
    y0: int
    x0: int
    y1: int
    x1: int
    touches_border: bool
    shape_key: bytes


def segments(g: np.ndarray, ignore: np.ndarray | None = None, max_segments: int = 400) -> list[Segment]:
    """4-connected same-colour components, background (largest) removed.

    Components are labelled with scipy.ndimage per colour and returned in
    raster order of their first pixel.
    """
    try:
        from scipy import ndimage
    except ImportError:          # pure-python fallback, same output order
        return _segments_py(g, ignore, max_segments)
    h, w = g.shape
    valid = np.ones((h, w), dtype=bool) if ignore is None else ~ignore
    found = []
    for c in np.unique(g[valid]):
        lab, n = ndimage.label((g == c) & valid, structure=_FOUR)
        if n == 0:
            continue
        flat = lab.ravel()
        order = np.argsort(flat, kind="stable")
        counts = np.bincount(flat, minlength=n + 1)
        bounds = np.cumsum(counts)
        for i in range(1, n + 1):
            idx = order[bounds[i - 1]:bounds[i]]
            found.append((int(idx[0]), int(c), idx))
    found.sort(key=lambda t: t[0])
    return _build_segments(found, h, w, max_segments)


def _build_segments(found, h, w, max_segments):
    out: list[Segment] = []
    for _, c, arr in found[:max_segments]:
        ys, xs = arr // w, arr % w
        y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
        my, mx = ys.mean(), xs.mean()
        k = int(np.argmin((ys - my) ** 2 + (xs - mx) ** 2))
        local = np.zeros((y1 - y0 + 1, x1 - x0 + 1), dtype=np.uint8)
        local[ys - y0, xs - x0] = 1
        shape_key = hashlib.blake2b(np.packbits(local).tobytes() + bytes(local.shape), digest_size=8).digest()
        out.append(Segment(c, len(arr), int(ys[k]), int(xs[k]), int(y0), int(x0), int(y1), int(x1),
                           bool(y0 == 0 or x0 == 0 or y1 == h - 1 or x1 == w - 1), shape_key))
    if out:
        big = max(out, key=lambda s: s.size)
        out = [s for s in out if s is not big]
    return out


def _segments_py(g, ignore, max_segments):
    h, w = g.shape
    seen = np.zeros((h, w), dtype=bool) if ignore is None else ignore.copy()
    flat = g.ravel()
    found = []
    for start in range(h * w):
        if seen.flat[start]:
            continue
        c = flat[start]
        stack, pix = [start], []
        seen.flat[start] = True
        while stack:
            p = stack.pop()
            pix.append(p)
            y, x = divmod(p, w)
            for q, ok in ((p - w, y > 0), (p + w, y < h - 1), (p - 1, x > 0), (p + 1, x < w - 1)):
                if ok and not seen.flat[q] and flat[q] == c:
                    seen.flat[q] = True
                    stack.append(q)
        found.append((start, int(c), np.sort(np.array(pix))))
    return _build_segments(found, h, w, max_segments)


_FOUR = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]])


def click_candidates(g: np.ndarray, ignore: np.ndarray | None = None, limit: int = 64) -> list[tuple[int, int, Segment]]:
    """One click per salient segment, ordered by salience (small, interior, rare colour first)."""
    segs = segments(g, ignore)
    if not segs:
        return []
    colour_area = np.bincount(g.ravel().astype(np.int64) & 15, minlength=16)

    def salience(s: Segment):
        return (s.touches_border, s.size > 400, colour_area[s.color] > 1200, s.size)

    segs.sort(key=salience)
    return [(s.cx, s.cy, s) for s in segs[:limit]]


In [ ]:
%%writefile /kaggle/working/arc3chimera/explorer.py
"""Control agent: graph exploration with novelty detection.

State graph over (masked) frame hashes. In every state the untried actions are
tried in priority order; when the current state is exhausted the agent walks
the known graph (BFS) to the nearest state that still has untried actions.
Transitions that ended in GAME_OVER are marked deadly and avoided.

The class is written so that a world model can re-rank untried actions
(``rank_untried``) and learn from every transition (``learn``); the plain
control uses a fixed salience order and learns nothing across levels.
"""
from __future__ import annotations

import random
from collections import deque
from dataclasses import dataclass, field

import numpy as np

from .perception import VolatilityTracker, click_candidates, grid_hash, last_grid

NOT_PLAYED, NOT_FINISHED, WIN, GAME_OVER = "NOT_PLAYED", "NOT_FINISHED", "WIN", "GAME_OVER"


@dataclass
class Obs:
    grid: np.ndarray
    state: str
    levels: int
    available: list[int]


@dataclass
class Node:
    grid: np.ndarray
    actions: list = field(default_factory=list)     # action keys in try order
    tier: dict = field(default_factory=dict)        # action key -> priority tier
    meta: dict = field(default_factory=dict)        # action key -> segment (for clicks)
    tried: set = field(default_factory=set)
    edges: dict = field(default_factory=dict)       # action key -> canonical successor
    deadly: set = field(default_factory=set)


def akey_to_call(ak):
    """Action key -> (action id, data dict) for the API."""
    if isinstance(ak, tuple):
        return 6, {"x": int(ak[1]), "y": int(ak[2])}
    return int(ak), {}


class GraphExplorer:
    name = "graph_explorer"

    def __init__(self, seed: int = 0, click_limit: int = 48):
        self.rng = random.Random(seed)
        self.click_limit = click_limit
        self.prev_obs: Obs | None = None
        self.prev_key = None
        self.prev_action = None
        self.plan: list = []
        self.level = -1
        self.stats = {"actions": 0, "resets": 0, "levels": [], "nodes_per_level": []}
        self._new_level()

    # ----- level bookkeeping -------------------------------------------------
    def _new_level(self):
        self.vol = VolatilityTracker()
        self.raw_grids: dict = {}      # raw hash -> grid
        self.log: list = []            # (raw_from, akey, raw_to, outcome)
        self.nodes: dict = {}          # canonical key -> Node
        self.canon_cache: dict = {}
        self.plan = []

    def canon(self, raw):
        k = self.canon_cache.get(raw)
        if k is None:
            k = grid_hash(self.raw_grids[raw], self.vol.mask)
            self.canon_cache[raw] = k
        return k

    def _rebuild(self):
        """Mask changed: re-key every stored grid and replay the transition log."""
        self.canon_cache = {}
        old = self.nodes
        self.nodes = {}
        for raw in self.raw_grids:
            self._ensure_node(self.canon(raw), self.raw_grids[raw], old_hint=None)
        for raw_from, ak, raw_to, outcome in self.log:
            self._record_edge(self.canon(raw_from), ak, None if raw_to is None else self.canon(raw_to), outcome)
        self.plan = []

    # ----- action generation -------------------------------------------------
    def candidate_actions(self, grid: np.ndarray, available: list[int]):
        acts, tiers, meta = [], {}, {}
        for a in available:
            if a in (1, 2, 3, 4, 5, 7):
                acts.append(a)
                tiers[a] = 0
        if 6 in available:
            for i, (x, y, seg) in enumerate(click_candidates(grid, self.vol.mask, self.click_limit)):
                ak = (6, x, y)
                acts.append(ak)
                tiers[ak] = 0 if i < 8 else (1 if i < 24 else 2)
                meta[ak] = seg
        return acts, tiers, meta

    def _ensure_node(self, key, grid, old_hint=None):
        n = self.nodes.get(key)
        if n is None:
            acts, tiers, meta = self.candidate_actions(grid, self._available)
            n = Node(grid=grid, actions=acts, tier=tiers, meta=meta)
            self.nodes[key] = n
        return n

    def _record_edge(self, key_from, ak, key_to, outcome):
        n = self.nodes.get(key_from)
        if n is None:
            return
        n.tried.add(ak)
        if outcome == "deadly":
            n.deadly.add(ak)
            n.edges.pop(ak, None)
        elif key_to is not None and key_to != key_from:
            n.edges[ak] = key_to

    # ----- hooks for world models ------------------------------------------
    def rank_untried(self, key, node: Node, untried: list) -> list:
        return sorted(untried, key=lambda a: (node.tier.get(a, 0), node.actions.index(a)))

    def state_value(self, key, node: Node, untried: list) -> float:
        """Lower is better: priority of visiting this frontier state."""
        return min(node.tier.get(a, 0) for a in untried)

    def learn(self, prev: Obs, ak, obs: Obs, outcome: str, key_from, key_to):
        pass

    def on_level_up(self, levels: int):
        pass

    # ----- main step ---------------------------------------------------------
    def untried(self, node: Node):
        return [a for a in node.actions if a not in node.tried]

    def act(self, obs: Obs):
        self._available = [a for a in obs.available if a != 0]
        # learn from the previous transition
        if self.prev_obs is not None and self.prev_action is not None:
            self._observe_transition(obs)
        if obs.levels != self.level:
            if self.level >= 0:
                self.stats["levels"].append(self.stats["actions"])
                self.stats["nodes_per_level"].append(len(self.nodes))
                self.on_level_up(obs.levels)
            self.level = obs.levels
            self._new_level()
        if obs.state in (NOT_PLAYED, GAME_OVER):
            self.prev_obs, self.prev_action = None, None
            self.plan = []
            self.stats["resets"] += 1
            return 0, {}
        raw = grid_hash(obs.grid)
        self.raw_grids.setdefault(raw, obs.grid)
        key = self.canon(raw)
        node = self._ensure_node(key, obs.grid)
        ak = self._choose(key, node)
        self.prev_obs, self.prev_key, self.prev_raw, self.prev_action = obs, key, raw, ak
        self.stats["actions"] += 1
        return akey_to_call(ak)

    def _observe_transition(self, obs: Obs):
        prev, ak = self.prev_obs, self.prev_action
        if obs.levels != prev.levels:
            outcome = "level_up"
        elif obs.state == GAME_OVER:
            outcome = "deadly"
        else:
            outcome = "step"
        raw_to = None
        if outcome == "step":
            raw_to = grid_hash(obs.grid)
            self.raw_grids.setdefault(raw_to, obs.grid)
            changed_mask = self.vol.update(prev.grid, obs.grid)
        else:
            changed_mask = False
        self.log.append((self.prev_raw, ak, raw_to, outcome))
        if changed_mask:
            self._rebuild()
        key_from = self.canon(self.prev_raw)
        key_to = None if raw_to is None else self.canon(raw_to)
        if key_to is not None:
            self._ensure_node(key_to, obs.grid)
        self._record_edge(key_from, ak, key_to, outcome)
        self.learn(prev, ak, obs, outcome, key_from, key_to)

    def _choose(self, key, node: Node):
        # follow an existing plan if it is still consistent
        if self.plan:
            nxt_key, ak = self.plan[0]
            if nxt_key == key:
                self.plan.pop(0)
                return ak
            self.plan = []
        u = self.untried(node)
        if u:
            return self.rank_untried(key, node, u)[0]
        path = self._path_to_frontier(key)
        if path:
            self.plan = path[1:]
            return path[0][1]
        safe = [a for a in node.actions if a not in node.deadly] or node.actions or [1]
        return self.rng.choice(safe)

    def _path_to_frontier(self, start):
        """BFS over known edges; returns [(state, action), ...] to the best frontier."""
        prev = {start: None}
        q = deque([start])
        best, best_score = None, None
        depth = {start: 0}
        while q:
            s = q.popleft()
            n = self.nodes.get(s)
            if n is None:
                continue
            if s != start:
                u = self.untried(n)
                if u:
                    score = (self.state_value(s, n, u), depth[s])
                    if best_score is None or score < best_score:
                        best, best_score = s, score
            if best_score is not None and depth[s] > best_score[1] + 6:
                break
            for ak, t in n.edges.items():
                if ak in n.deadly or t in prev:
                    continue
                prev[t] = (s, ak)
                depth[t] = depth[s] + 1
                q.append(t)
        if best is None:
            return None
        steps = []
        cur = best
        while prev[cur] is not None:
            s, ak = prev[cur]
            steps.append((s, ak))
            cur = s
        steps.reverse()
        return steps


In [ ]:
%%writefile /kaggle/working/arc3chimera/chimera.py
"""CHIMERA world model for ARC-AGI-3 (numpy port of the CHIMERA principle).

CHIMERA (Agnuxo1/ARC2_CHIMERA) computes with cellular-automaton texture
updates and a holographic memory in OpenGL shaders. The Kaggle runtime has no
display, so the same two ingredients are ported to numpy here:

* Holographic memory: every context feature f has a fixed random phase vector
  exp(i*phi_f) of dimension D. An action context is the superposition of its
  feature vectors (one complex trace). Three traces store what happened after
  each action: EFFECT (state changed / no-op), GOAL (level completed) and
  DEADLY (game over). Writes use the delta rule, reads are the normalised real
  correlation, so similar contexts share what was learned.
* Cellular-automaton rule table: the 3x3 neighbourhood around a click and the
  action id form a local rule key; its observed outcomes are written into the
  same traces as one more feature, i.e. local rules generalise across states.

The memory persists across levels of a game (the control forgets everything at
each level). The world model only re-ranks which untried action to try next
and which frontier state to visit; exploration stays complete, so an untried
action predicted useless is postponed, never removed.
"""
from __future__ import annotations

import hashlib

import numpy as np

from .explorer import GraphExplorer, Node


def _phase_vector(feature: tuple, dim: int, salt: int = 0) -> np.ndarray:
    seed = int.from_bytes(hashlib.blake2b(repr((salt, feature)).encode(), digest_size=8).digest(), "little")
    rng = np.random.default_rng(seed)
    return np.exp(1j * rng.uniform(0, 2 * np.pi, dim)).astype(np.complex64)


def _patch(grid: np.ndarray, y: int, x: int, r: int) -> np.ndarray:
    """(2r+1)^2 neighbourhood of (y, x), -1 outside the screen: a local CA rule context."""
    p = np.full((2 * r + 1, 2 * r + 1), -1, dtype=np.int8)
    y0, y1, x0, x1 = max(0, y - r), min(grid.shape[0], y + r + 1), max(0, x - r), min(grid.shape[1], x + r + 1)
    p[y0 - y + r:y1 - y + r, x0 - x + r:x1 - x + r] = grid[y0:y1, x0:x1]
    return p


class HolographicMemory:
    """Complex phase associative memory with delta-rule writes."""

    def __init__(self, dim: int = 512, lr: float = 0.5, salt: int = 0):
        self.dim = dim
        self.salt = salt
        self.lr = lr
        self.trace = np.zeros(dim, dtype=np.complex64)
        self.cache: dict = {}
        self.writes = 0

    def key(self, feats: list) -> np.ndarray:
        k = np.zeros(self.dim, dtype=np.complex64)
        for f in feats:
            v = self.cache.get(f)
            if v is None:
                v = _phase_vector(f, self.dim, self.salt)
                self.cache[f] = v
            k += v
        return k / max(1, len(feats))

    def read(self, k: np.ndarray) -> float:
        return float(np.real(np.vdot(k, self.trace)) / self.dim)

    def write(self, k: np.ndarray, target: float):
        err = target - self.read(k)
        norm = float(np.real(np.vdot(k, k))) / self.dim or 1.0
        self.trace += (self.lr * err / norm) * k
        self.writes += 1


class ChimeraAgent(GraphExplorer):
    name = "chimera"

    def __init__(self, seed: int = 0, dim: int = 512, alpha: float = 2.0, persist: bool = True,
                 use_ca: bool = True, use_goal: bool = True, use_dead: bool = True, use_kbca: bool = False,
                 salt_seed: bool = False, **kw):
        super().__init__(seed=seed, **kw)
        self.dim, self.alpha, self.persist = dim, alpha, persist
        self.use_ca, self.use_goal, self.use_dead = use_ca, use_goal, use_dead
        self.use_kbca = use_kbca
        self.salt = seed if salt_seed else 0
        self.kb_loc: dict = {}
        self._init_memory()
        self.stats.update({"wm_reorders": 0, "wm_writes": 0})

    def _init_memory(self):
        self.effect = HolographicMemory(self.dim, salt=self.salt)
        self.goal = HolographicMemory(self.dim, salt=self.salt)
        self.dead = HolographicMemory(self.dim, salt=self.salt)
        self.fcache: dict = {}

    # ----- features ----------------------------------------------------------
    def features(self, grid: np.ndarray, ak, seg=None) -> list:
        if not isinstance(ak, tuple):
            f = [("a", ak)]
            loc = self.kb_loc.get(ak) if self.use_kbca else None
            if loc is not None:
                f.append(("kb", ak, _patch(grid, loc[0], loc[1], 3).tobytes()))
            return f
        _, x, y = ak
        f = [("click",)]
        if seg is not None:
            f += [("c", seg.color), ("cs", seg.color, seg.shape_key), ("sz", seg.color, min(seg.size, 63) // 8),
                  ("bd", seg.touches_border)]
        else:
            f.append(("c", int(grid[y, x])))
        if self.use_ca:
            y0, y1, x0, x1 = max(0, y - 1), min(64, y + 2), max(0, x - 1), min(64, x + 2)
            patch = np.full((3, 3), -1, dtype=np.int8)
            patch[y0 - y + 1:y1 - y + 1, x0 - x + 1:x1 - x + 1] = grid[y0:y1, x0:x1]
            f.append(("ca", patch.tobytes()))
        return f

    def _key(self, node_key, node: Node, ak):
        ck = (node_key, ak, self.kb_loc.get(ak) if self.use_kbca and not isinstance(ak, tuple) else None)
        k = self.fcache.get(ck)
        if k is None:
            k = self.effect.key(self.features(node.grid, ak, node.meta.get(ak)))
            self.fcache[ck] = k
        return k

    def value(self, node_key, node: Node, ak) -> float:
        k = self._key(node_key, node, ak)
        v = self.effect.read(k)
        if self.use_goal:
            v += 2.0 * max(0.0, self.goal.read(k))
        if self.use_dead:
            v -= 2.0 * max(0.0, self.dead.read(k))
        return v

    # ----- hooks -------------------------------------------------------------
    def rank_untried(self, key, node: Node, untried: list) -> list:
        base = super().rank_untried(key, node, untried)
        if self.effect.writes == 0:
            return base
        order = {a: i for i, a in enumerate(base)}
        ranked = sorted(untried, key=lambda a: (node.tier.get(a, 0) - self.alpha * self.value(key, node, a), order[a]))
        if ranked[0] != base[0]:
            self.stats["wm_reorders"] += 1
        return ranked

    def state_value(self, key, node: Node, untried: list) -> float:
        if self.effect.writes == 0:
            return super().state_value(key, node, untried)
        return min(node.tier.get(a, 0) - self.alpha * self.value(key, node, a) for a in untried)

    def learn(self, prev, ak, obs, outcome, key_from, key_to):
        node = self.nodes.get(key_from)
        if node is None:
            return
        k = self._key(key_from, node, ak)
        changed = outcome != "step" or (key_to is not None and key_to != key_from)
        if self.use_kbca and not isinstance(ak, tuple) and outcome == "step" and changed:
            d = (prev.grid != obs.grid) & ~self.vol.mask
            if d.any():
                ys, xs = np.nonzero(d)
                self.kb_loc[ak] = (int(ys.mean()), int(xs.mean()))
        self.effect.write(k, 1.0 if changed else -1.0)
        if self.use_goal:
            self.goal.write(k, 1.0 if outcome == "level_up" else 0.0)
        if self.use_dead:
            self.dead.write(k, 1.0 if outcome == "deadly" else 0.0)
        self.stats["wm_writes"] += 1

    def _rebuild(self):
        self.fcache = {}
        super()._rebuild()

    def _new_level(self):
        super()._new_level()
        self.fcache = {}
        if not getattr(self, "persist", True) and hasattr(self, "effect"):
            self._init_memory()


In [ ]:
%%writefile /kaggle/working/arc3chimera/baselines.py
"""Random baseline equivalent to the official sample agent (no RESET except when required)."""
import random

import numpy as np


class RandomAgent:
    name = "random"

    def __init__(self, seed: int = 0):
        self.rng = random.Random(seed)
        self.stats = {}

    def act(self, obs):
        if obs.state in ("NOT_PLAYED", "GAME_OVER"):
            return 0, {}
        a = self.rng.choice([x for x in obs.available if x != 0])
        if a == 6:
            colours = np.unique(obs.grid).tolist()
            c = self.rng.choice(colours)
            ys, xs = np.where(obs.grid == c)
            i = self.rng.randrange(len(xs))
            return 6, {"x": int(xs[i]), "y": int(ys[i])}
        return a, {}


In [ ]:
import os, sys, time, json, logging, threading
from concurrent.futures import ThreadPoolExecutor
sys.path.insert(0, '/kaggle/working')
import pandas as pd

import subprocess
RERUN = os.getenv('KAGGLE_IS_COMPETITION_RERUN', '').strip().lower() not in ('', '0', 'false', 'no')
if RERUN:
    # wait for the competition gateway (fixed command, no user input)
    rc = subprocess.run(['curl', '--fail', '--silent', '--retry', '999', '--retry-all-errors', '--retry-delay', '5',
                         '--retry-max-time', '900', '-o', '/dev/null', 'http://gateway:8001/api/games']).returncode
    print('gateway wait, curl rc =', rc, flush=True)
    if rc != 0:
        print('WARNING: gateway did not answer /api/games; the game list below is retried before failing', flush=True)
mode, envdir = ('online', '') if RERUN else ('offline', '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/')
with open('/kaggle/working/.env', 'w') as f:
    f.write(f"""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE={mode}
ENVIRONMENTS_DIR={envdir}
{'RECORDINGS_DIR=/kaggle/working/server_recording' if RERUN else ''}
""")
import dotenv
dotenv.load_dotenv(dotenv_path='/kaggle/working/.env', override=True)
import arc_agi
from arcengine import GameAction, GameState
from arc3chimera.explorer import Obs
from arc3chimera.perception import last_grid
from arc3chimera.runner import make_policy
logging.disable(logging.INFO)

POLICY, KW = 'chimera', {"use_kbca": True, "alpha": 4.0}
MAX_ACTIONS = 30000 if RERUN else 2000
T_START = T_KERNEL
TOTAL_BUDGET = 7.5 * 3600          # whole notebook must finish in 9 h: margin for setup and teardown
WORKERS = 8
arcade = arc_agi.Arcade()
games = [e.game_id for e in arcade.available_environments]
for _ in range(10):                # the gateway may still be warming up
    if games or not RERUN:
        break
    time.sleep(30)
    arcade = arc_agi.Arcade()
    games = [e.game_id for e in arcade.available_environments]
if not games:
    raise RuntimeError(f'no games available from the Arcade (rerun={RERUN}); refusing to finish silently')
PER_GAME = max(120.0, (TOTAL_BUDGET - (time.time() - T_START)) * WORKERS / max(1, len(games)))
print(len(games), 'games; per-game budget s =', PER_GAME, flush=True)

def play(game_id):
    t0 = time.time()
    n, lv, nones = 0, 0, 0
    try:
        env = arcade.make(game_id)
        r = env.observation_space
        if r is None:
            r = env.step(GameAction.RESET, {})
        pol = make_policy(POLICY, 0, **KW)
        while r is not None and r.state != GameState.WIN and n < MAX_ACTIONS:
            if time.time() - t0 > PER_GAME or time.time() - T_START > TOTAL_BUDGET:
                break
            obs = Obs(grid=last_grid(r.frame), state=r.state.name, levels=r.levels_completed,
                      available=list(r.available_actions))
            aid, data = pol.act(obs)
            nxt = env.step(GameAction.from_id(aid), data)
            n += 1
            if nxt is None and nones < 3:    # transient failure: resync with a RESET, at most 3 times
                nones += 1
                nxt = env.step(GameAction.RESET, {})
                pol.prev_obs, pol.prev_action, pol.plan = None, None, []
            r = nxt
            if r is not None:
                lv = r.levels_completed
    except Exception as e:
        return 'error', f'{game_id}: ERROR {e!r} after {n} actions'
    status = 'ok' if r is not None else 'none'
    return status, f'{game_id}: {status} {n} actions, levels={lv}, none_resets={nones}, {time.time()-t0:.0f}s'

counts = {'ok': 0, 'none': 0, 'error': 0}
with ThreadPoolExecutor(WORKERS) as ex:
    for status, line in ex.map(play, games):
        counts[status] += 1
        print(line, flush=True)
print(f'games={len(games)} ok={counts["ok"]} ended_on_none={counts["none"]} errors={counts["error"]} '
      f'elapsed={time.time() - T_START:.0f}s since kernel start', flush=True)

if not RERUN:
    sc = arcade.get_scorecard()
    print('Score', sc.score, 'levels', sc.total_levels_completed, '/', sc.total_levels, 'actions', sc.total_actions)
    for e in sc.environments:
        print(e.id, round(e.score, 3), e.levels_completed, e.actions)
    pd.DataFrame(data=[['1_0', '1', True, 1]], columns=['row_id', 'game_id', 'end_of_game', 'score']).to_parquet('/kaggle/working/submission.parquet', index=False)
